# General Analysis

This sections is to have a general analysis that includes not only the target authors, but also data from the coauthors extracted

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (10, 5)})

DATA_DIR = "/home/ramir713/PublicationAnalysis/data"

filtered_all_df = pd.read_csv(f"{DATA_DIR}/filtered_all_df.csv")
filtered_all_df_corr = pd.read_csv(f"{DATA_DIR}/filtered_all_df_corr.csv")

print("Loaded all-authors tables:")
print(f"filtered_all_df      : {filtered_all_df.shape}")
print(f"filtered_all_df_corr : {filtered_all_df_corr.shape}")

papers_per_author = filtered_all_df.groupby("queried_author").size().sort_values(ascending=False)
author_counts = (
    filtered_all_df_corr.groupby("queried_author")
    .agg(
        total_papers=("article_id", "count"),
        avg_citations=("total_citations", "mean"),
    )
    .sort_values("total_papers", ascending=False)
    .reset_index()
)

display(author_counts.head(15))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(author_counts["queried_author"].head(15)[::-1], author_counts["total_papers"].head(15)[::-1], color="steelblue", edgecolor="white")
ax.set_title("All authors: papers produced as corresponding authors")
ax.set_xlabel("Number of papers")
plt.tight_layout()
plt.show()

all_corr = filtered_all_df_corr.assign(citation_group="corresponding")
all_non = filtered_all_df.loc[~filtered_all_df["is_corresponding_author"]].assign(citation_group="non-corresponding")
combined = pd.concat([all_corr, all_non], ignore_index=True)

citations_by_group = combined.groupby("citation_group")["total_citations"].agg(["count", "mean", "median"])
display(citations_by_group)

## All Authors Analysis

This notebook uses `filtered_all_df.csv` and `filtered_all_df_corr.csv` to analyze the full author group, including corresponding and non-corresponding papers.

In [ ]:
# Overall all-authors summaries
all_author_counts = (
    filtered_all_df.groupby("queried_author")
    .agg(
        total_papers=("article_id", "nunique"),
        total_citations=("total_citations", "sum"),
        avg_citations=("total_citations", "mean"),
    )
    .sort_values(["total_papers", "total_citations"], ascending=False)
    .reset_index()
)

all_author_corr_counts = (
    filtered_all_df_corr.groupby("queried_author")
    .agg(
        corresponding_papers=("article_id", "nunique"),
        corresponding_citations=("total_citations", "sum"),
        corresponding_avg_citations=("total_citations", "mean"),
    )
    .reset_index()
)

all_author_summary = all_author_counts.merge(all_author_corr_counts, on="queried_author", how="left")
all_author_summary = all_author_summary.fillna(0)

display(all_author_summary.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = all_author_summary.sort_values("total_papers", ascending=True).tail(15)
ax.barh(plot_df["queried_author"], plot_df["total_papers"], color="steelblue", edgecolor="white")
ax.set_title("All authors: total papers in the filtered dataset")
ax.set_xlabel("Number of papers")
plt.tight_layout()
plt.show()

In [ ]:
# Corresponding vs non-corresponding comparison
all_df_non = filtered_all_df.loc[~filtered_all_df["is_corresponding_author"]].copy()

comparison_df = pd.DataFrame(
    {
        "group": ["corresponding", "non_corresponding"],
        "papers": [len(filtered_all_df_corr), len(all_df_non)],
        "total_citations": [filtered_all_df_corr["total_citations"].sum(), all_df_non["total_citations"].sum()],
        "avg_citations": [filtered_all_df_corr["total_citations"].mean(), all_df_non["total_citations"].mean()],
    }
)

display(comparison_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparison_df["group"], comparison_df["avg_citations"], color=["steelblue", "coral"], edgecolor="white")
ax.set_title("All authors: average citations by author role")
ax.set_ylabel("Average citations")
plt.tight_layout()
plt.show()